## <b>损失聚合策略决定了每个样本或每个token对总梯度的贡献</b>。主流方法有两种：
- 序列级聚合（Sequence-level）：先计算每个响应序列的平均损失，再对整个批次的序列损失求平均。这是<b>GRPO采用的方法</b>。<b>这种方法给予每个响应同等的权重，无论其长短。</b>
- 令牌级聚合（Token-level）：将批次中所有响应的所有token的损失加起来，再除以总token数。这是<b>DAPO采用的方法</b>。这种方法保证了每个token对总梯度的贡献是相等的。

Yu等人（2025）指出，<b>序列级聚合存在一个缺陷</b>：<b>它会降低 长响应 中每个token的影响力，可能导致模型偏向于生成 更短 的答案，并且无法充分从 长而复杂 的正确推理中学习</b>

### 令牌级损失聚合是否适用于所有场景？
#### 与序列级计算相比，令牌级损失在Base模型上更有效，但在Instruct模型上改进有限。

作者在Qwen3-8B的<b>Base</b>和<b>Aligned</b>版本上，对这两种聚合策略进行了系统性比较。
- <b>对于Base模型</b>：<b>令牌级损失聚合</b>在收敛速度、峰值精度和鲁棒性上都全面优于序列级聚合，尤其是在困难数据集上。这验证了<b>令牌级聚合能确保每个token（尤其是长序列中的token）都得到平等优化的猜想</b>。

- <b>对于Aligned模型</b>：结果再次反转。<b>序列级聚合</b>在大多数数据集和设置下，都比令牌级聚合表现得更好。

<b>为什么会这样？</b>作者分析，Aligned模型本身已经具备了强大而稳定的推理能力，强制让每个token的梯度贡献均等化，反而可能是非必要甚至有害的。在这种情况下，序列级聚合更好地保留了高质量、对齐输出的整体结构和一致性。

这个发现再次强调了“没有免费的午餐”这一原则。<b>最优的技术选择，深度依赖于你的初始模型状态。</b>
- 对于从零开始训练的Base模型，<b>令牌级聚合是更好的选择</b>；
- 而对于已经良好对齐的Instruct模型，<b>序列级聚合可能更优。</b>